# EasyRAG Query Modes

This notebook compares the five built-in retrieval modes in EasyRAG: `naive`, `local`, `global`, `hybrid`, and `mix`. The corpus is intentionally small and deterministic so that you can focus on how the retrieval paths are assembled rather than on provider setup.

## What you will do

- create a small architecture-style corpus
- index it into a temporary EasyRAG workspace
- run the same question through all five retrieval modes
- compare citation titles, surfaced entities, surfaced relations, and metadata

## What to keep in mind

On small corpora, several modes may converge to similar results. That is expected. The point of this notebook is not to force dramatic differences. The point is to make each mode's role visible and inspectable.


## Step 1: Import the public API and runtime helper

We only need the high-level `EasyRAG` API, `QueryParam`, and `run_sync`. The rest of the notebook builds deterministic stubs and a tiny corpus around them.


In [ ]:
import json
import tempfile

from easyrag import EasyRAG, QueryParam
from easyrag.support.async_utils import run_sync


This cell should not produce meaningful output. Its job is to define the public interface surface we will use in the rest of the notebook.


## Step 2: Define deterministic helper stubs

To keep the notebook zero-key runnable, we replace external providers with deterministic stubs. The embedding stub produces simple keyword-count vectors, the query-model stub keeps rewrite and MQE predictable, and the KG stub creates a small graph around a billing-service example.


In [ ]:
_KEYWORDS = [
    "billing",
    "service",
    "gateway",
    "datastore",
    "pricing",
    "request",
    "policy",
    "routes",
    "depends",
    "customer",
    "subscription",
]


def _stub_embedding(texts: list[str]) -> list[list[float]]:
    vectors = []
    for text in texts:
        lowered = text.lower()
        vector = [float(lowered.count(keyword)) for keyword in _KEYWORDS]
        vector.append(float(len(lowered.split())))
        vectors.append(vector)
    return vectors


def _stub_query_model(prompt: str, *, task: str, count: int = 1) -> str | list[str]:
    cleaned = prompt.split(":", 1)[-1].strip()
    if task == "rewrite":
        return cleaned
    if task == "mqe":
        return [f"{cleaned} variant {index}" for index in range(1, count + 1)]
    raise ValueError(task)


def _stub_kg_model(text: str, *, entity_types, max_entities: int, max_relations: int, metadata=None):
    del entity_types, metadata
    lowered = text.lower()
    entities = []
    relations = []
    if "billing service" in lowered:
        entities.append({"name": "Billing Service", "type": "service", "description": "Handles billing flows."})
    if "api gateway" in lowered:
        entities.append({"name": "API Gateway", "type": "service", "description": "Ingress layer."})
    if "customer datastore" in lowered:
        entities.append({"name": "Customer Datastore", "type": "dependency", "description": "Customer records store."})
    if "pricing policy" in lowered:
        entities.append({"name": "Pricing Policy", "type": "concept", "description": "Pricing rules."})

    names = {item["name"] for item in entities}
    if {"Billing Service", "API Gateway"} <= names:
        relations.append({"source": "API Gateway", "target": "Billing Service", "relation": "routes_to", "description": "Gateway routes billing traffic."})
    if {"Billing Service", "Customer Datastore"} <= names:
        relations.append({"source": "Billing Service", "target": "Customer Datastore", "relation": "depends_on", "description": "Billing Service depends on Customer Datastore."})
    if {"Billing Service", "Pricing Policy"} <= names:
        relations.append({"source": "Billing Service", "target": "Pricing Policy", "relation": "uses", "description": "Billing Service uses Pricing Policy."})

    return {
        "entities": entities[:max_entities],
        "relations": relations[:max_relations],
    }


This cell only defines helper behavior, so it should not print anything. The important thing is that the same text now produces stable vectors, stable query preprocessing behavior, and stable graph extraction results.


## Step 3: Create a small corpus with connected concepts

The corpus includes a few related records about a billing service, an API gateway, a pricing policy, and a datastore. This is enough structure for the graph-aware modes to have something to work with without making the notebook noisy.


In [ ]:
demo_texts = [
    "# Billing Service\nBilling Service computes invoices. Billing Service depends on Customer Datastore. Billing Service uses Pricing Policy.\n",
    "# API Gateway\nAPI Gateway receives requests and routes them to Billing Service.\n",
    "# Pricing\nPricing Policy defines plan rules and discount thresholds.\n",
    "# Datastore\nCustomer Datastore stores customer subscription history.\n",
]

demo_ids = [
    "doc::billing",
    "doc::gateway",
    "doc::pricing",
    "doc::datastore",
]

demo_paths = [
    "docs/billing.md",
    "docs/gateway.md",
    "docs/pricing.md",
    "docs/datastore.md",
]

print(json.dumps({"ids": demo_ids, "paths": demo_paths}, indent=2))


You should see four logical document paths. These paths later help make citations easier to interpret.


## Step 4: Create and initialize the workspace

We use a temporary working directory so the comparison stays isolated. The injected helper stubs keep the run deterministic while still exercising the same public API that a configured application would use.


In [ ]:
temp_dir = tempfile.TemporaryDirectory()
working_dir = temp_dir.name
workspace = "query-modes"

rag = EasyRAG(
    working_dir=working_dir,
    workspace=workspace,
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
    llm_model_func=_stub_kg_model,
)

run_sync(rag.initialize_storages())
print(json.dumps({"working_dir": working_dir, "workspace": workspace}, indent=2))


You should see a temporary workspace path. The workspace is empty at this stage; the next step is what turns the corpus into indexed retrieval state.


## Step 5: Insert the corpus into the workspace

The insertion step creates chunk, summary, vector, and graph-aware records in one lifecycle. That is important because the retrieval modes we compare later all operate on the same indexed workspace, not on separate corpora.


In [ ]:
insert_stats = run_sync(rag.ainsert(demo_texts, ids=demo_ids, file_paths=demo_paths))
print(json.dumps(insert_stats, indent=2))


A successful output should show nonzero counts for documents, chunks, entities, and relations. That confirms the notebook has enough indexed structure to make the mode comparison meaningful.


## Step 6: Run one question across all five retrieval modes

We use the same question for all modes and keep rewrite and MQE disabled. That lets the comparison focus on the mode differences rather than on preprocessing variation.


In [ ]:
question = "How do billing requests reach the service?"
modes = ["naive", "local", "global", "hybrid", "mix"]
mode_results = {}

for mode in modes:
    result = run_sync(
        rag.aquery(
            question,
            QueryParam(
                mode=mode,
                chunk_top_k=2,
                rewrite_enabled=False,
                mqe_enabled=False,
            ),
        )
    )
    mode_results[mode] = {
        "citation_titles": [item["title"] for item in result.citations],
        "entities": result.entities[:5],
        "relations": [item["relation"] for item in result.relations[:3]],
        "metadata": {
            "vector_backend": result.metadata.get("vector_backend"),
            "chunk_strategies": result.metadata.get("chunk_strategies"),
        },
    }

print(json.dumps(mode_results, indent=2))


You should see a compact side-by-side comparison for all five modes. On this corpus, several modes may return overlapping citation titles. That is normal. The useful observation is which entities and relations become visible once graph-aware retrieval enters the picture.


## Step 7: Inspect one mode in a more grounded way

A comparison table is useful, but it is also worth reading one actual citation payload. Here we inspect the `local` mode because it is the clearest bridge between entity-centric retrieval and grounded chunk output.


In [ ]:
local_result = run_sync(
    rag.aquery(
        question,
        QueryParam(mode="local", chunk_top_k=2, rewrite_enabled=False, mqe_enabled=False),
    )
)

print(json.dumps({
    "citations": local_result.citations,
    "entities": local_result.entities,
    "relations": local_result.relations,
}, indent=2))


In this view, the relationship between graph-aware retrieval and grounded output becomes easier to see. The relations and surfaced entities are not the final answer themselves. They are retrieval signals that help explain why certain citations appear.


## Step 8: Clean up the temporary workspace

We close the storages and remove the temporary directory so the notebook leaves no persistent state behind.


In [ ]:
run_sync(rag.finalize_storages())
temp_dir.cleanup()

print("Closed storages and removed the temporary query-modes workspace.")


After cleanup, the mode comparison is complete. The important takeaway is not that one mode always wins. The important takeaway is that EasyRAG exposes multiple retrieval perspectives over the same indexed knowledge.

## Next steps

- read [docs/04-retrieval-overview.md](../../docs/04-retrieval-overview.md) for the architecture-level explanation
- run [04_knowledge_graph.ipynb](04_knowledge_graph.ipynb) to focus on automatically extracted graph signals
- revisit [00_quickstart.ipynb](00_quickstart.ipynb) if you want the smaller end-to-end loop again
